In [ ]:
"""
BO+NN + CDU Benchmark Suite — Q1 Journal Version (CDU Extension)
================================================================
Constrained Dual Unrolling (CDU) replaces the soft-penalty MLP with
two coupled networks that embed primal-descent / dual-ascent dynamics
as inductive biases, following 2025-2026 CDU literature.

Architecture
------------
  Primal net  π_θ : (z ∈ ℝⁿ, λ ∈ ℝᵐ)  →  x ∈ [lb, ub]   (sigmoid-scaled)
  Dual   net  δ_φ : (x ∈ ℝⁿ, λ ∈ ℝᵐ)  →  λ' ∈ ℝᵐ₊        (softplus output)

CDU Unrolling (K steps)
-----------------------
  λ₀  =  0
  for k = 0 … K-1:
      xₖ   =  π_θ(z, λₖ)          # primal descent step (learned)
      λₖ₊₁ =  δ_φ(xₖ, λₖ)         # dual   ascent step  (learned)

Loss (Augmented Lagrangian form)
---------------------------------
  ℒ = log((f(x_K)+M)²+ε)          # log-compressed objective
    + λ_K · relu(g(x_K))           # Lagrangian dual term
    + (ρ/2) ‖relu(g(x_K))‖²       # quadratic augmentation

  Box constraints satisfied BY CONSTRUCTION via sigmoid scaling of π_θ.

New Optuna hyperparameters (per trial)
--------------------------------------
  K   ∈ CDU_K_CHOICES   — unrolling depth
  ρ   ∈ RHO_CHOICES     — augmented Lagrangian penalty
  lr  ∈ LR_CHOICES      — Adam learning rate
  z_i ∈ [-1, 1]         — latent input vector

Why CDU enhances the existing framework
----------------------------------------
  1. Faster convergence  : K learned primal-dual steps ≫ many soft-penalty
                           gradient steps.
  2. Strict feasibility  : λ_K dynamically amplifies violated constraints,
                           avoiding manual penalty tuning.
  3. Complementarity     : BO (Optuna) tunes K, ρ, lr jointly — replacing
                           fragile fixed penalty weights.
  4. OOD generalization  : Monotone primal-descent / dual-ascent inductive
                           bias keeps near-feasible solutions across z.

Output files
------------
  figures/01_convergence_curves.png
  figures/02_relative_gap.png
  figures/03_feasibility_analysis.png
  figures/04_comparison_summary.png
  figures/05_variable_distribution.png

  results/01_main_results.csv
  results/02_feasibility_analysis.csv
  results/03_relative_gap.csv
  results/04_statistical_robustness.csv
  results/04b_variable_distribution.csv
  results/05_convergence_data.csv
  results/06_initialization_comparison.csv
"""

# ─── Suppress warnings ────────────────────────────────────────────────────────
import warnings, os, time, csv
warnings.filterwarnings("ignore")
os.makedirs("figures", exist_ok=True)
os.makedirs("results",  exist_ok=True)

# ─── Core imports ─────────────────────────────────────────────────────────────
import jax
import jax.numpy as jnp
from jax import jit
import optax
import optuna
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tqdm import tqdm

optuna.logging.set_verbosity(optuna.logging.WARNING)
jax.config.update("jax_enable_x64", True)

# ══════════════════════════════════════════════════════════════════════════════
#  GLOBAL CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
HIDDEN         = 32
N_STEPS        = 150
N_TRIALS_MAIN  = 1000
N_TRIALS_STAT  = 1000
N_STAT_RUNS    = 30
BO_FEAS_TOL    = 0.0   # strict: any g_i(x) > 0 => infeasible

# ── CDU-specific hyperparameter search spaces ─────────────────────────────────
# K  : number of primal-dual unrolling steps (Optuna tunes per trial)
# ρ  : augmented Lagrangian quadratic penalty coefficient
CDU_K_CHOICES  = [3, 5, 8]           # unrolling depth choices
RHO_CHOICES    = [1e2, 1e3, 1e4]     # augmented Lagrangian ρ choices
LR_CHOICES     = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2]

# log((f+M)² + ε) numerical safety
LOG_EPS        = 1e-30

# ─── Publication plot style ───────────────────────────────────────────────────
GRID_KW = dict(alpha=0.30, linestyle='--', color='#999999')
plt.rcParams.update({
    'figure.facecolor': 'white',   'axes.facecolor':   'white',
    'axes.edgecolor':   '#444444', 'axes.linewidth':   0.8,
    'text.color':       '#111111', 'axes.labelcolor':  '#111111',
    'xtick.color':      '#333333', 'ytick.color':      '#333333',
    'grid.color':       '#999999', 'grid.linestyle':   '--',
    'grid.alpha':       0.30,
    'font.family':      'serif',   'font.size':        10,
    'axes.titlesize':   11,        'axes.labelsize':   10,
    'xtick.labelsize':  9,         'ytick.labelsize':  9,
    'legend.fontsize':  9,         'legend.framealpha': 0.92,
    'legend.edgecolor': '#888888', 'figure.dpi':       150,
    'savefig.dpi':      300,       'savefig.bbox':     'tight',
    'lines.linewidth':  1.6,
})
C_BO    = '#2166ac'   # BO+CDU  — blue
C_FIXED = '#d6604d'   # IPOPT   — red-orange


# ══════════════════════════════════════════════════════════════════════════════
#  CDU NETWORK INFRASTRUCTURE
# ══════════════════════════════════════════════════════════════════════════════

def init_primal_params(key, n_vars: int, n_cons: int):
    """
    Glorot-init weights for the primal network π_θ.
    Input  : [z (n_vars)  ‖  λ (n_cons)]  →  hidden  →  x (n_vars)
    Output : x mapped through sigmoid to [lb, ub] inside CDU loss.
    """
    n_in = n_vars + n_cons
    k = jax.random.split(key, 6)
    return [
        jax.random.normal(k[0], (n_in,   HIDDEN)) * jnp.sqrt(2.0 / n_in),   jnp.zeros(HIDDEN),
        jax.random.normal(k[1], (HIDDEN,  HIDDEN)) * jnp.sqrt(2.0 / HIDDEN), jnp.zeros(HIDDEN),
        jax.random.normal(k[2], (HIDDEN,  n_vars)) * jnp.sqrt(2.0 / HIDDEN), jnp.zeros(n_vars),
    ]


def init_dual_params(key, n_vars: int, n_cons: int):
    """
    Glorot-init weights for the dual network δ_φ.
    Input  : [x (n_vars)  ‖  λ (n_cons)]  →  hidden  →  λ' (n_cons)
    Output : softplus-activated to enforce λ' ≥ 0 (dual feasibility).
    """
    n_in = n_vars + n_cons
    k = jax.random.split(key, 6)
    return [
        jax.random.normal(k[0], (n_in,   HIDDEN)) * jnp.sqrt(2.0 / n_in),   jnp.zeros(HIDDEN),
        jax.random.normal(k[1], (HIDDEN,  HIDDEN)) * jnp.sqrt(2.0 / HIDDEN), jnp.zeros(HIDDEN),
        jax.random.normal(k[2], (HIDDEN,  n_cons)) * jnp.sqrt(2.0 / HIDDEN), jnp.zeros(n_cons),
    ]


def init_cdu_params(key, n_vars: int, n_cons: int) -> dict:
    """Return a JAX-pytree dict holding both primal and dual parameters."""
    k1, k2 = jax.random.split(key)
    return {
        'primal': init_primal_params(k1, n_vars, n_cons),
        'dual':   init_dual_params(k2,   n_vars, n_cons),
    }


# ─── CDU loss cache: maps (prob_key, K, rho) → compiled loss_fn ───────────────
_CDU_LOSS_CACHE: dict = {}

def get_cdu_loss(prob_key: str, K: int, rho: float):
    """
    Return (and cache) a CDU loss function for (problem, K, rho).
    Caching ensures the same Python object id is reused, so the JIT
    trainer cache hits correctly without recompilation.
    """
    cache_key = (prob_key, K, rho)
    if cache_key not in _CDU_LOSS_CACHE:
        p = PROBLEMS[prob_key]
        _CDU_LOSS_CACHE[cache_key] = _make_cdu_loss(
            obj_fn   = p['obj_fn'],
            cons_fn  = p['cons_fn'],
            lb_j     = p['lb_j'],
            ub_j     = p['ub_j'],
            n_vars   = p['n_vars'],
            n_cons   = p['n_cons'],
            M        = p['M'],
            K        = K,
            rho      = rho,
        )
    return _CDU_LOSS_CACHE[cache_key]


def _make_cdu_loss(obj_fn, cons_fn, lb_j, ub_j, n_vars, n_cons, M, K, rho):
    """
    Factory: builds a CDU loss closure for fixed (K, ρ).

    The returned loss_fn(params, z) runs K unrolling steps:
      x_k   = sigmoid_scaled(primal_net(z ‖ λ_k))   — primal descent
      λ_{k+1} = softplus(dual_net(x_k ‖ λ_k))       — dual ascent

    Then computes:
      ℒ = log_loss(f(x_K), M)
        + λ_K · relu(g(x_K))
        + (ρ/2) ‖relu(g(x_K))‖²

    Box constraints [lb, ub] are satisfied BY CONSTRUCTION via sigmoid
    scaling inside the primal network — no additional box penalty needed.
    """
    def loss_fn(params, z):
        pp = params['primal']   # primal network weights
        dp = params['dual']     # dual   network weights

        # ── Initialise multipliers ─────────────────────────────────────────
        lam = jnp.zeros(n_cons)

        # ── K primal-dual unrolling steps ──────────────────────────────────
        # Python for-loop: JAX traces & unrolls statically at JIT time.
        x = lb_j  # placeholder; overwritten in first iteration
        for _ in range(K):
            # Primal step: π_θ(z ‖ λ) → x ∈ [lb, ub]
            inp_p = jnp.concatenate([z, lam])
            h = jax.nn.relu(jnp.dot(inp_p,  pp[0]) + pp[1])
            h = jax.nn.relu(jnp.dot(h,       pp[2]) + pp[3])
            x = lb_j + jax.nn.sigmoid(jnp.dot(h, pp[4]) + pp[5]) * (ub_j - lb_j)

            # Dual step: δ_φ(x ‖ λ) → λ' ≥ 0  (softplus enforces non-negativity)
            inp_d = jnp.concatenate([x, lam])
            h = jax.nn.relu(jnp.dot(inp_d,  dp[0]) + dp[1])
            h = jax.nn.relu(jnp.dot(h,       dp[2]) + dp[3])
            lam = jax.nn.softplus(jnp.dot(h, dp[4]) + dp[5])

        # ── Augmented Lagrangian loss ───────────────────────────────────────
        f    = obj_fn(x)
        g    = cons_fn(x)
        rv   = jax.nn.relu(g)                          # relu(g): violations only

        aug_lag = (log_loss(f, M)                      # log-compressed objective
                   + jnp.dot(lam, rv)                  # Lagrangian dual term
                   + 0.5 * rho * jnp.sum(rv ** 2))    # quadratic augmentation

        pen = jnp.sum(rv)   # total constraint violation (for logging)
        return aug_lag, (f, pen, x)

    return loss_fn


# ─── Trainer cache ─────────────────────────────────────────────────────────────
_trainer_cache: dict = {}

def get_scan_trainer(loss_fn, lr: float, n_steps: int):
    """
    Return (train_fn, optimizer) for (loss_fn, lr); compile once.
    params is a pytree dict {'primal': [...], 'dual': [...]}.
    z is the fixed latent input passed per-trial by Optuna.
    """
    cache_key = (id(loss_fn), lr, n_steps)
    if cache_key not in _trainer_cache:
        optimizer = optax.adam(lr)

        @jit
        def train_fn(params, z, opt_state):
            """
            Run n_steps of Adam on (primal + dual) params for fixed latent z.
            Returns updated params and per-step history of (loss, f, pen, x).
            """
            @jit
            def scan_body(carry, _):
                params, opt_state = carry
                (loss, (f_val, pen, x)), grads = jax.value_and_grad(
                    loss_fn, argnums=0, has_aux=True)(params, z)
                updates, new_state = optimizer.update(grads, opt_state, params)
                new_params = optax.apply_updates(params, updates)
                return (new_params, new_state), (loss, f_val, pen, x)

            (final_params, final_opt_state), hist = jax.lax.scan(
                scan_body, (params, opt_state), None, length=n_steps)
            return (final_params, final_opt_state), hist

        _trainer_cache[cache_key] = (train_fn, optimizer)
    return _trainer_cache[cache_key]


def warmup_jit():
    """
    Pre-compile all (problem × K × ρ × lr) JIT graphs before timing starts.
    CDU requires one compilation per (K, ρ, lr) triple per problem since
    JAX statically unrolls the Python for-loop over K.
    """
    total = len(PROBLEM_ORDER) * len(CDU_K_CHOICES) * len(RHO_CHOICES) * len(LR_CHOICES)
    print(f"  Pre-compiling {total} JIT graphs (problem × K × ρ × lr) ...")
    for pk in PROBLEM_ORDER:
        p = PROBLEMS[pk]
        key = jax.random.PRNGKey(0)
        params = init_cdu_params(key, p['n_vars'], p['n_cons'])
        z      = jnp.zeros(p['n_vars'])
        for K in CDU_K_CHOICES:
            for rho in RHO_CHOICES:
                loss_fn = get_cdu_loss(pk, K, rho)
                for lr in LR_CHOICES:
                    tf, opt = get_scan_trainer(loss_fn, lr, N_STEPS)
                    out = tf(params, z, opt.init(params))
                    jax.block_until_ready(out[0])
        print(f"    OK  {DISPLAY_NAMES[pk]}", flush=True)
    print()


# ══════════════════════════════════════════════════════════════════════════════
#  LOG-LOSS HELPER
#  loss_obj = log((f + M)² + ε)
#
#  Gradient:  2(f+M) / ((f+M)² + ε)  ≈  2/(f+M)  for large |f+M|
#  => objective gradient shrinks as |f+M| grows, so the Lagrangian dual
#     term (λ_K · relu(g)) always dominates near any violated constraint,
#     regardless of how negative f is.
#
#  REQUIREMENT: f(x) + M > 0  for all x in the search box.
#  M is chosen conservatively per-problem (see PROBLEMS dict).
# ══════════════════════════════════════════════════════════════════════════════
def log_loss(f, M):
    """log((f+M)² + ε)  — numerically safe, gradient-friendly."""
    return jnp.log((f + M) ** 2 + LOG_EPS)


# ══════════════════════════════════════════════════════════════════════════════
#  PROBLEM DEFINITIONS  (objective + constraint functions only;
#  loss_fn is now constructed generically by _make_cdu_loss)
# ══════════════════════════════════════════════════════════════════════════════

# ── 1. Branin ──────────────────────────────────────────────────────────────────
_BR_LB = jnp.array([-5.,  0.]);  _BR_UB = jnp.array([10., 15.])

@jit
def branin_obj(x):
    a,b,c,r,s,t = 1., 5.1/(4*jnp.pi**2), 5./jnp.pi, 6., 10., 1./(8*jnp.pi)
    return a*(x[1]-b*x[0]**2+c*x[0]-r)**2 + s*(1-t)*jnp.cos(x[0]) + s

@jit
def branin_cons(x):
    return jnp.stack([0.5 - jnp.sin(x[0]+x[1])**2])

# ── 2. Himmelblau ──────────────────────────────────────────────────────────────
_HB_LB = jnp.array([-6., -6.]);  _HB_UB = jnp.array([6., 6.])

@jit
def himmelblau_obj(x):
    return (x[0]**2 + x[1] - 11.)**2 + (x[0] + x[1]**2 - 7.)**2

@jit
def himmelblau_cons(x):
    return jnp.stack([x[0]**2 + x[1] - 4., x[0] + x[1]**2 - 3.])

# ── 3. Constrained Rosenbrock ──────────────────────────────────────────────────
_RB_LB = jnp.array([-5., -5.]);  _RB_UB = jnp.array([5., 5.])

@jit
def rosenbrock_obj(x):
    return 100.*(x[1] - x[0]**2)**2 + (1. - x[0])**2

@jit
def rosenbrock_cons(x):
    return jnp.stack([x[0]**2 + x[1]**2 - 2., x[0] + x[1] - 1.])

# ── 4. Pressure Vessel ─────────────────────────────────────────────────────────
_PV_LB = jnp.array([0.0625, 0.0625, 10., 10.])
_PV_UB = jnp.array([99.,    99.,   200., 200.])

@jit
def pressure_vessel_obj(x):
    return (0.6224*x[0]*x[2]*x[3] + 1.7781*x[1]*x[2]**2
            + 3.1661*x[0]**2*x[3] + 19.84*x[0]**2*x[2])

@jit
def pressure_vessel_cons(x):
    return jnp.stack([-x[0]+0.0193*x[2],
                       -x[1]+0.00954*x[2],
                       jnp.pi*x[2]**2*x[3]+(4./3.)*jnp.pi*x[2]**3-1_296_000.])

# ── 5. G04 – Gear Train ────────────────────────────────────────────────────────
_G4_LB = jnp.array([78., 33., 27., 27., 27.])
_G4_UB = jnp.array([102., 45., 45., 45., 45.])

@jit
def g04_obj(x):
    return 5.3578457*x[2]**2 + 0.8356891*x[0]*x[4] + 37.293239*x[0] - 40792.141

@jit
def g04_cons(x):
    C1 = 85.334407+0.0056858*x[1]*x[4]+0.0006262*x[0]*x[3]-0.0022053*x[2]*x[4]
    C2 = 80.51249 +0.0071317*x[1]*x[2]+0.002995*x[0]*x[1] +0.0021813*x[2]**2
    C3 = 9.300961 +0.0047026*x[2]*x[4]+0.0012547*x[0]*x[2]+0.0019085*x[2]*x[3]
    return jnp.stack([C1-92., 85.-C1, C2-110., 90.-C2, C3-25., 20.-C3])

# ── 6. G06 ─────────────────────────────────────────────────────────────────────
_G6_LB = jnp.array([13.,  0.]);  _G6_UB = jnp.array([100., 100.])

@jit
def g06_obj(x):
    return (x[0] - 10.)**3 + (x[1] - 20.)**3

@jit
def g06_cons(x):
    return jnp.stack([-(x[0]-5.)**2-(x[1]-5.)**2+100.,
                       (x[0]-6.)**2+(x[1]-5.)**2-82.81])

# ── 7. Welded Beam ─────────────────────────────────────────────────────────────
_WB_LB = jnp.array([0.1, 0.1, 0.1, 0.1])
_WB_UB = jnp.array([2., 10., 10., 2.])

@jit
def welded_beam_obj(x):
    return 1.10471*x[0]**2*x[1] + 0.04811*x[2]*x[3]*(14. + x[1])

@jit
def welded_beam_cons(x):
    h,l,t,b = x[0],x[1],x[2],x[3]
    P=6000.;L=14.;E=30.e6;G=12.e6
    tp  = P/(jnp.sqrt(2.)*h*l)
    R   = jnp.sqrt(l**2/4.+((h+t)/2.)**2)
    J   = 2.*jnp.sqrt(2.)*h*l*(l**2/12.+((h+t)/2.)**2)
    tpp = P*L*R/J
    tau = jnp.sqrt(tp**2+tpp**2)
    sig = 6.*P*L/(b*t**2)
    dlt = 4.*P*L**3/(E*b*t**3)
    Pc  = (4.013*E*jnp.sqrt(t**2*b**6/36.)/L**2
           *(1.-(t/(2.*L))*jnp.sqrt(E/(4.*G))))
    return jnp.stack([tau-13600., sig-30000., dlt-0.25, P-Pc, h-b])


# ══════════════════════════════════════════════════════════════════════════════
#  PROBLEM REGISTRY
#  New CDU fields:
#    n_cons   — number of inequality constraints (needed for dual net input dim)
#    M        — log-loss offset; must satisfy f(x)+M > 0 everywhere
#  (No per-problem loss_fn needed: CDU loss is built generically by get_cdu_loss)
# ══════════════════════════════════════════════════════════════════════════════
PROBLEMS = {
    "Branin": dict(
        n_vars=2, n_cons=1,
        lb=np.array([-5., 0.]), ub=np.array([10., 15.]),
        lb_j=_BR_LB, ub_j=_BR_UB,
        obj_fn=branin_obj, cons_fn=branin_cons,
        M=50.,      # f ∈ [0.4, 308]  => f+M > 0 ✓
        g_labels=['0.5-sin²(x₁+x₂)'], var_names=['x1', 'x2'],
    ),
    "Himmelblau": dict(
        n_vars=2, n_cons=2,
        lb=np.array([-6., -6.]), ub=np.array([6., 6.]),
        lb_j=_HB_LB, ub_j=_HB_UB,
        obj_fn=himmelblau_obj, cons_fn=himmelblau_cons,
        M=10.,      # f ∈ [0, 890]    => f+M > 0 ✓
        g_labels=['x1²+x2-4', 'x1+x2²-3'], var_names=['x1', 'x2'],
    ),
    "Rosenbrock": dict(
        n_vars=2, n_cons=2,
        lb=np.array([-5., -5.]), ub=np.array([5., 5.]),
        lb_j=_RB_LB, ub_j=_RB_UB,
        obj_fn=rosenbrock_obj, cons_fn=rosenbrock_cons,
        M=5.,       # f ≥ 0           => f+M > 0 ✓
        g_labels=['x1²+x2²-2', 'x1+x2-1'], var_names=['x1', 'x2'],
    ),
    "PressureVessel": dict(
        n_vars=4, n_cons=3,
        lb=np.array([0.0625, 0.0625, 10., 10.]),
        ub=np.array([99., 99., 200., 200.]),
        lb_j=_PV_LB, ub_j=_PV_UB,
        obj_fn=pressure_vessel_obj, cons_fn=pressure_vessel_cons,
        M=100.,     # f ∈ [180, 1.2e7] => f+M > 0 ✓
        g_labels=['-x1+0.0193x3', '-x2+0.00954x3',
                  'πx3²x4+(4/3)πx3³-1296000'],
        var_names=['x1', 'x2', 'x3', 'x4'],
    ),
    "GearTrain": dict(
        n_vars=5, n_cons=6,
        lb=np.array([78., 33., 27., 27., 27.]),
        ub=np.array([102., 45., 45., 45., 45.]),
        lb_j=_G4_LB, ub_j=_G4_UB,
        obj_fn=g04_obj, cons_fn=g04_cons,
        M=40000.,   # f ≥ -32218      => f+M ≥ 7782 > 0 ✓
        g_labels=['C1-92', '85-C1', 'C2-110', '90-C2', 'C3-25', '20-C3'],
        var_names=['x1', 'x2', 'x3', 'x4', 'x5'],
    ),
    "G06": dict(
        n_vars=2, n_cons=2,
        lb=np.array([13., 0.]), ub=np.array([100., 100.]),
        lb_j=_G6_LB, ub_j=_G6_UB,
        obj_fn=g06_obj, cons_fn=g06_cons,
        M=10000.,   # f ≥ -7973       => f+M ≥ 2027 > 0 ✓
        g_labels=['-(x1-5)²-(x2-5)²+100', '(x1-6)²+(x2-5)²-82.81'],
        var_names=['x1', 'x2'],
    ),
    "WeldedBeam": dict(
        n_vars=4, n_cons=5,
        lb=np.array([0.1, 0.1, 0.1, 0.1]),
        ub=np.array([2., 10., 10., 2.]),
        lb_j=_WB_LB, ub_j=_WB_UB,
        obj_fn=welded_beam_obj, cons_fn=welded_beam_cons,
        M=5.,       # f ≥ 1.4         => f+M > 0 ✓
        g_labels=['τ-13600', 'σ-30000', 'δ-0.25', 'P-Pc', 'h-b'],
        var_names=['h', 'l', 't', 'b'],
    ),
}

PROBLEM_ORDER = ["Branin", "Himmelblau", "Rosenbrock",
                 "PressureVessel", "GearTrain", "G06", "WeldedBeam"]
DISPLAY_NAMES = {
    "Branin":        "Branin",
    "Himmelblau":    "Himmelblau",
    "Rosenbrock":    "Constrained Rosenbrock",
    "PressureVessel":"Pressure Vessel",
    "GearTrain":     "G04 Gear Train",
    "G06":           "G06 Global Opt.",
    "WeldedBeam":    "Welded Beam",
}


# ══════════════════════════════════════════════════════════════════════════════
#  RELATIVE OPTIMALITY GAP
# ══════════════════════════════════════════════════════════════════════════════
_EPS = 1e-12

def relative_gap(f_a, f_b):
    if any(np.isinf(v) or np.isnan(v) for v in [f_a, f_b]):
        return np.nan
    return abs(f_a - f_b) / max(abs(f_b), _EPS)


# ══════════════════════════════════════════════════════════════════════════════
#  CDU STUDY RUNNER
#  Optuna now jointly tunes:  K (unrolling depth), ρ (aug. Lagrangian),
#                              lr (Adam rate), z (latent input)
# ══════════════════════════════════════════════════════════════════════════════
def run_study(prob_key: str, obj_fn, cons_fn, n_vars, n_cons,
              n_trials, n_steps,
              jax_seed=42, problem_name="", optuna_seed=42, show_pbar=True):
    """
    CDU version of run_study.
    Returns: study, record, fmask, best_x, best_f, best_fhist
    """
    record       = dict(bx=[], bl=[])
    best_f_live  = [np.inf]
    best_fhist   = [None]
    n_feas_count = [0]

    if show_pbar:
        pbar = tqdm(total=n_trials, desc=f"  {problem_name:<32}",
                    unit="trial", colour="blue",
                    bar_format="{l_bar}{bar:30}{r_bar}", dynamic_ncols=True)

    def objective(trial):
        # ── Optuna suggests CDU hyperparameters ───────────────────────────
        lr  = trial.suggest_categorical('lr',  LR_CHOICES)
        K   = trial.suggest_categorical('K',   CDU_K_CHOICES)
        rho = trial.suggest_categorical('rho', RHO_CHOICES)
        z   = jnp.array([trial.suggest_float(f"z{i}", -1., 1.)
                          for i in range(n_vars)])

        # ── Build / retrieve CDU loss for this (K, rho) ───────────────────
        loss_fn = get_cdu_loss(prob_key, K, rho)

        key       = jax.random.PRNGKey(jax_seed * 10000 + trial.number)
        params    = init_cdu_params(key, n_vars, n_cons)
        train_fn, opt = get_scan_trainer(loss_fn, lr, n_steps)
        opt_state = opt.init(params)

        # ── Train: gradient descent on primal + dual params jointly ───────
        (final_params, _), hist = train_fn(params, z, opt_state)
        losses, f_vals, penalties, xs = hist
        jax.block_until_ready(losses)

        best_idx  = int(jnp.argmin(losses))
        best_x    = np.asarray(xs[best_idx])
        best_loss = float(losses[best_idx])

        # ── Strict feasibility check ──────────────────────────────────────
        raw_cv  = np.array(cons_fn(jnp.array(best_x)))
        is_feas = bool(np.all(raw_cv <= 0.0))

        record['bx'].append(best_x)
        record['bl'].append(best_loss)

        if is_feas:
            fv = float(obj_fn(jnp.array(best_x)))
            if fv < best_f_live[0]:
                best_f_live[0] = fv
                best_fhist[0]  = np.array(f_vals)
            n_feas_count[0] += 1

        if show_pbar:
            bs = f"{best_f_live[0]:.4f}" if not np.isinf(best_f_live[0]) else "---"
            pbar.set_postfix(
                feasible=f"{n_feas_count[0]}/{trial.number+1}",
                best_f=bs, K=K, rho=f"{rho:.0e}", refresh=False)
            pbar.update(1)
        return best_loss

    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=optuna_seed))
    study.optimize(objective, n_trials=n_trials)
    if show_pbar:
        pbar.close()

    record['bx']  = np.array(record['bx'])
    record['blc'] = np.minimum.accumulate(record['bl'])

    # Best feasible solution (strict: all g_i ≤ 0)
    sols   = record['bx']
    cv_all = jax.vmap(cons_fn)(jnp.array(sols))
    fmask  = np.array(jnp.all(cv_all <= 0.0, axis=-1))
    feas_s = sols[fmask]

    if len(feas_s):
        fobjs  = np.array(jax.vmap(obj_fn)(jnp.array(feas_s)))
        best_x = feas_s[int(np.argmin(fobjs))]
        best_f = float(np.min(fobjs))
    else:
        best_x = sols[int(np.argmin(record['bl']))]
        best_f = np.inf

    return study, record, fmask, best_x, best_f, best_fhist[0]


# ══════════════════════════════════════════════════════════════════════════════
#  IPOPT SOLVER — fixed initialisation [1, ..., 1]  (unchanged)
# ══════════════════════════════════════════════════════════════════════════════
def _build_casadi_nlp(prob_key):
    import casadi as ca
    xv = ca.SX.sym('x', PROBLEMS[prob_key]['n_vars'])

    if prob_key == "Branin":
        a,b,c,r,s,t = 1., 5.1/(4*np.pi**2), 5/np.pi, 6., 10., 1/(8*np.pi)
        f  = a*(xv[1]-b*xv[0]**2+c*xv[0]-r)**2+s*(1-t)*ca.cos(xv[0])+s
        g  = [0.5 - ca.sin(xv[0]+xv[1])**2]
        lb = [-ca.inf];  ub = [0]
    elif prob_key == "Himmelblau":
        f  = (xv[0]**2+xv[1]-11)**2+(xv[0]+xv[1]**2-7)**2
        g  = [xv[0]**2+xv[1]-4, xv[0]+xv[1]**2-3]
        lb = [-ca.inf]*2;  ub = [0]*2
    elif prob_key == "Rosenbrock":
        f  = 100*(xv[1]-xv[0]**2)**2+(1-xv[0])**2
        g  = [xv[0]**2+xv[1]**2-2, xv[0]+xv[1]-1]
        lb = [-ca.inf]*2;  ub = [0]*2
    elif prob_key == "PressureVessel":
        x1,x2,x3,x4 = xv[0],xv[1],xv[2],xv[3]
        f  = (0.6224*x1*x3*x4+1.7781*x2*x3**2+3.1661*x1**2*x4+19.84*x1**2*x3)
        g  = [-x1+0.0193*x3, -x2+0.00954*x3,
              ca.pi*x3**2*x4+(4/3)*ca.pi*x3**3-1296000]
        lb = [-ca.inf]*3;  ub = [0]*3
    elif prob_key == "GearTrain":
        x1,x2,x3,x4,x5 = xv[0],xv[1],xv[2],xv[3],xv[4]
        f  = 5.3578457*x3**2+0.8356891*x1*x5+37.293239*x1-40792.141
        C1 = 85.334407+0.0056858*x2*x5+0.0006262*x1*x4-0.0022053*x3*x5
        C2 = 80.51249 +0.0071317*x2*x3+0.002995*x1*x2 +0.0021813*x3**2
        C3 = 9.300961 +0.0047026*x3*x5+0.0012547*x1*x3+0.0019085*x3*x4
        g  = [C1-92, 85-C1, C2-110, 90-C2, C3-25, 20-C3]
        lb = [-ca.inf]*6;  ub = [0]*6
    elif prob_key == "G06":
        f  = (xv[0]-10)**3+(xv[1]-20)**3
        g  = [-(xv[0]-5)**2-(xv[1]-5)**2+100, (xv[0]-6)**2+(xv[1]-5)**2-82.81]
        lb = [-ca.inf]*2;  ub = [0]*2
    elif prob_key == "WeldedBeam":
        h,l,t,b = xv[0],xv[1],xv[2],xv[3]
        P=6000; L=14; E=30e6; G=12e6
        tp  = P/(ca.sqrt(2)*h*l)
        R   = ca.sqrt(l**2/4+((h+t)/2)**2)
        J   = 2*ca.sqrt(2)*h*l*(l**2/12+((h+t)/2)**2)
        tpp = P*L*R/J
        tau = ca.sqrt(tp**2+tpp**2)
        sig = 6*P*L/(b*t**2)
        dlt = 4*P*L**3/(E*b*t**3)
        Pc  = 4.013*E*ca.sqrt(t**2*b**6/36)/L**2*(1-t/(2*L)*ca.sqrt(E/(4*G)))
        f   = 1.10471*h**2*l+0.04811*t*b*(14+l)
        g   = [tau-13600, sig-30000, dlt-0.25, P-Pc, h-b]
        lb  = [-ca.inf]*5;  ub = [0]*5
    else:
        raise ValueError(f"Unknown problem: {prob_key}")

    return xv, f, g, lb, ub


def solve_ipopt_fixed(prob_key: str):
    """Single IPOPT solve from [1,...,1].  Strict feasibility: all g_i ≤ 0."""
    try:
        import casadi as ca
        p  = PROBLEMS[prob_key]
        n  = p['n_vars']
        xv, f_ca, g_ca, lbg, ubg = _build_casadi_nlp(prob_key)
        opts   = {'ipopt': {'print_level': 0, 'tol': 1e-10, 'max_iter': 3000},
                  'print_time': 0}
        solver = ca.nlpsol('S', 'ipopt',
                           {'x': xv, 'f': f_ca, 'g': ca.vertcat(*g_ca)}, opts)
        g_fn   = ca.Function('g', [xv], [ca.vertcat(*g_ca)])

        x0 = np.ones(n).tolist()
        t0 = time.perf_counter()
        try:
            sol  = solver(x0=x0,
                          lbx=p['lb'].tolist(), ubx=p['ub'].tolist(),
                          lbg=lbg, ubg=ubg)
            xs   = np.array(sol['x']).flatten()
            fs   = float(sol['f'])
            cv   = np.array(g_fn(xs)).flatten()
            feas = bool(np.all(cv <= 0.0))
        except Exception:
            xs = np.full(n, np.nan);  fs = np.inf
            cv = np.full(len(g_ca), np.nan);  feas = False

        wall_ms = (time.perf_counter() - t0) * 1000
        return fs, xs, cv, wall_ms, feas

    except ImportError:
        n = PROBLEMS[prob_key]['n_vars']
        return np.inf, np.full(n, np.nan), np.array([np.nan]), 0., False


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN EXPERIMENT LOOP
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("  BO+NN+CDU vs IPOPT (fixed init) — Journal Benchmark Suite")
print("  CDU Loss: log((f+M)²+ε) + λ_K·relu(g) + (ρ/2)‖relu(g)‖²")
print("  Optuna tunes: K (unrolling depth), ρ (aug. Lagrangian), lr, z")
print("=" * 70)
print(f"  N_TRIALS_MAIN={N_TRIALS_MAIN}  N_STEPS={N_STEPS}")
print(f"  CDU_K_CHOICES={CDU_K_CHOICES}  RHO_CHOICES={RHO_CHOICES}")
print(f"  LR_CHOICES={LR_CHOICES}")
print(f"  Feasibility: strict g_i(x) <= 0.0 for all i\n")

warmup_jit()

main_results = {}
stat_results = {}

for prob_key in PROBLEM_ORDER:
    p     = PROBLEMS[prob_key]
    dname = DISPLAY_NAMES[prob_key]
    print(f"\n{'---'*23}\n  [{prob_key}]  {dname}\n{'---'*23}")

    # ── A. Main BO+CDU run ────────────────────────────────────────────────────
    print(f"  >  Main BO+CDU ({N_TRIALS_MAIN} trials) ...")
    t0 = time.perf_counter()
    study, record, fmask, bo_x, bo_f, bo_fhist = run_study(
        prob_key=prob_key,
        obj_fn=p['obj_fn'], cons_fn=p['cons_fn'],
        n_vars=p['n_vars'], n_cons=p['n_cons'],
        n_trials=N_TRIALS_MAIN, n_steps=N_STEPS,
        jax_seed=42, problem_name=dname, optuna_seed=42)
    bo_time = (time.perf_counter() - t0) * 1000
    bo_cv   = np.array(p['cons_fn'](jnp.array(bo_x)))
    bo_feas = bool(np.all(bo_cv <= 0.0))
    bo_fr   = fmask.mean() * 100
    bo_tcv  = float(np.sum(np.maximum(bo_cv, 0.)))
    print(f"  OK  f={bo_f:.5f}  feasible={bo_feas}  "
          f"feas_rate={bo_fr:.0f}%  time={bo_time:.0f}ms")

    # Best CDU trial hyperparameters (for reporting)
    best_trial = study.best_trial
    best_K   = best_trial.params.get('K',   '?')
    best_rho = best_trial.params.get('rho', '?')
    best_lr  = best_trial.params.get('lr',  '?')
    print(f"       Best CDU trial: K={best_K}  ρ={best_rho:.0e}  lr={best_lr}")

    # ── B. IPOPT fixed init ───────────────────────────────────────────────────
    print(f"  >  IPOPT fixed [1,...,1] ...")
    fi_f, fi_x, fi_cv, fi_time, fi_feas = solve_ipopt_fixed(prob_key)
    fi_tcv = (float(np.sum(np.maximum(fi_cv, 0.)))
              if not np.all(np.isnan(fi_cv)) else np.nan)
    rel_gap_bo = relative_gap(bo_f, fi_f)
    rel_gap_fi = relative_gap(fi_f, bo_f)
    if not np.isinf(fi_f):
        print(f"  OK  f={fi_f:.5f}  feasible={fi_feas}  time={fi_time:.0f}ms")
        print(f"      Rel gaps: BO+CDU={rel_gap_bo:.4f}  IPOPT={rel_gap_fi:.4f}")
    else:
        print("  XX  IPOPT: CasADi not available or solver failed.")

    main_results[prob_key] = dict(
        bo=dict(best_f=bo_f, best_x=bo_x, cv=bo_cv, feasible=bo_feas,
                feas_rate=bo_fr, total_cv=bo_tcv, time_ms=bo_time,
                fmask=fmask, record=record, fhist=bo_fhist,
                rel_gap=rel_gap_bo,
                best_K=best_K, best_rho=best_rho, best_lr=best_lr),
        ipopt=dict(best_f=fi_f, best_x=fi_x, cv=fi_cv, feasible=fi_feas,
                   total_cv=fi_tcv, time_ms=fi_time, rel_gap=rel_gap_fi),
    )

    # ── C. Statistical robustness ─────────────────────────────────────────────
    print(f"  >  Statistical analysis ({N_STAT_RUNS} × {N_TRIALS_STAT} trials) ...")
    stat_rows = []
    for seed in tqdm(range(N_STAT_RUNS),
                     desc=f"  {'Stats: '+dname:<32}",
                     unit="run", colour="green",
                     bar_format="{l_bar}{bar:28}{r_bar}", dynamic_ncols=True):
        ts = time.perf_counter()
        _, srec, sfmask, sbx, sbf, _ = run_study(
            prob_key=prob_key,
            obj_fn=p['obj_fn'], cons_fn=p['cons_fn'],
            n_vars=p['n_vars'], n_cons=p['n_cons'],
            n_trials=N_TRIALS_STAT, n_steps=N_STEPS,
            jax_seed=seed, problem_name="", optuna_seed=42, show_pbar=False)
        run_t = (time.perf_counter() - ts) * 1000
        scv   = np.array(p['cons_fn'](jnp.array(sbx)))
        sfeas = bool(np.all(scv <= 0.0))
        stat_rows.append(dict(
            problem=prob_key, seed=seed,
            best_f=sbf, best_x=sbx.tolist(),
            is_feasible=int(sfeas),
            feas_rate=sfmask.mean() * 100,
            total_cv=float(np.sum(np.maximum(scv, 0.))),
            time_ms=run_t,
        ))

    stat_results[prob_key] = stat_rows
    valid_fs = [r['best_f'] for r in stat_rows
                if r['is_feasible'] and not np.isinf(r['best_f'])]
    print(f"  OK  feasible={sum(r['is_feasible'] for r in stat_rows)}/{N_STAT_RUNS}"
          + (f"  mean_f={np.mean(valid_fs):.5f}  std_f={np.std(valid_fs):.5f}"
             if valid_fs else "  (no feasible)"))


# ══════════════════════════════════════════════════════════════════════════════
#  STATISTICAL SUMMARY HELPER
# ══════════════════════════════════════════════════════════════════════════════
def stat_summary(pk):
    rows   = stat_results[pk]
    feas   = [r for r in rows if r['is_feasible'] and not np.isinf(r['best_f'])]
    fs     = [r['best_f'] for r in feas]
    all_x  = np.array([r['best_x'] for r in rows])
    feas_x = np.array([r['best_x'] for r in feas])

    var_mean_all  = all_x.mean(axis=0)
    var_std_all   = all_x.std(axis=0)
    var_mean_feas = (feas_x.mean(axis=0) if len(feas_x)
                     else np.full(all_x.shape[1], np.nan))
    var_std_feas  = (feas_x.std(axis=0)  if len(feas_x)
                     else np.full(all_x.shape[1], np.nan))

    return dict(
        n_feas        = len(feas),
        feas_rate     = np.mean([r['is_feasible'] for r in rows]) * 100,
        mean_f        = np.mean(fs)  if fs else np.nan,
        std_f         = np.std(fs)   if fs else np.nan,
        var_f         = np.var(fs)   if fs else np.nan,
        mean_tcv      = np.mean([r['total_cv'] for r in rows]),
        var_mean_all  = var_mean_all,
        var_std_all   = var_std_all,
        var_mean_feas = var_mean_feas,
        var_std_feas  = var_std_feas,
    )


# ══════════════════════════════════════════════════════════════════════════════
#  SAVE CSV RESULTS
# ══════════════════════════════════════════════════════════════════════════════
print("\n  >  Saving CSV results ...")

def _fstr(v):
    return f"{v:.6f}" if (not np.isinf(v) and not np.isnan(v)) else "N/A"

def _xstr(x):
    return ";".join(f"{v:.5f}" for v in x) if not np.all(np.isnan(x)) else "N/A"

# 01. Main results  (added CDU-specific columns: Best_K, Best_rho, Best_lr)
with open("results/01_main_results.csv", "w", newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f)
    w.writerow(["Problem", "Method", "Best_f",
                "Feasible", "Total_CV", "Feas_Rate_pct",
                "Rel_Gap_wrt_other", "Time_ms",
                "Best_K", "Best_rho", "Best_lr",    # CDU hyperparams
                "Best_x"])
    for pk in PROBLEM_ORDER:
        bo = main_results[pk]['bo']
        ip = main_results[pk]['ipopt']
        for tag, d, k_, r_, l_ in [
            ("BO+CDU",     bo, bo.get('best_K','N/A'), bo.get('best_rho','N/A'), bo.get('best_lr','N/A')),
            ("IPOPT_fixed",ip, 'N/A','N/A','N/A'),
        ]:
            fr = d.get('feas_rate', 'N/A')
            w.writerow([pk, tag, _fstr(d['best_f']),
                        d['feasible'],
                        _fstr(d['total_cv']) if not np.isnan(d['total_cv']) else "N/A",
                        f"{fr:.1f}" if isinstance(fr, float) else fr,
                        _fstr(d['rel_gap']),
                        f"{d['time_ms']:.0f}",
                        k_, r_, l_,
                        _xstr(d['best_x'])])

# 02. Feasibility analysis
with open("results/02_feasibility_analysis.csv", "w", newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f)
    w.writerow(["Problem", "Method", "Constraint_Index", "Constraint_Label",
                "g_i_value", "Violation_max_gi_0",
                "Constraint_Feasible", "Total_CV", "Solution_Feasible"])
    for pk in PROBLEM_ORDER:
        p = PROBLEMS[pk]
        for tag, d in [("BO+CDU",     main_results[pk]['bo']),
                       ("IPOPT_fixed", main_results[pk]['ipopt'])]:
            cv_arr = np.atleast_1d(d['cv'])
            is_nan = np.all(np.isnan(cv_arr))
            total  = d['total_cv']
            for idx, lbl in enumerate(p['g_labels']):
                gv   = float(cv_arr[idx]) if not is_nan else np.nan
                viol = max(gv, 0.) if not np.isnan(gv) else np.nan
                fi   = ("yes" if gv <= 0.0 else "VIOLATED") if not np.isnan(gv) else "N/A"
                w.writerow([pk, tag, idx+1, lbl,
                            f"{gv:.10g}"   if not np.isnan(gv)   else "N/A",
                            f"{viol:.6e}"  if not np.isnan(viol) else "N/A",
                            fi,
                            f"{total:.6e}" if not np.isnan(total) else "N/A",
                            "yes" if d['feasible'] else "VIOLATED"])

# 03. Relative gap  (updated Method label to BO+CDU)
with open("results/03_relative_gap.csv", "w", newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f)
    w.writerow(["Problem",
                "BO_CDU_f", "IPOPT_f",
                "Rel_gap_BO_CDU_wrt_IPOPT",
                "Rel_gap_IPOPT_wrt_BO_CDU",
                "BO_CDU_feasible", "IPOPT_feasible",
                "Stat_mean_f_BO_CDU", "Stat_std_f_BO_CDU", "Stat_var_f_BO_CDU",
                "Stat_feas_rate_pct"])
    for pk in PROBLEM_ORDER:
        bo = main_results[pk]['bo'];  ip = main_results[pk]['ipopt']
        ss = stat_summary(pk)
        w.writerow([pk,
                    _fstr(bo['best_f']), _fstr(ip['best_f']),
                    _fstr(bo['rel_gap']), _fstr(ip['rel_gap']),
                    bo['feasible'], ip['feasible'],
                    _fstr(ss['mean_f']), _fstr(ss['std_f']), _fstr(ss['var_f']),
                    f"{ss['feas_rate']:.1f}"])

# 04. Statistical robustness
with open("results/04_statistical_robustness.csv", "w", newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f)
    max_vars = max(PROBLEMS[pk]['n_vars'] for pk in PROBLEM_ORDER)
    var_cols  = [f"x{i+1}" for i in range(max_vars)]
    w.writerow(["Problem", "Seed", "Best_f",
                "Is_Feasible", "Feas_Rate_pct", "Total_CV", "Time_ms"]
               + var_cols)
    for pk in PROBLEM_ORDER:
        n = PROBLEMS[pk]['n_vars']
        for r in stat_results[pk]:
            xrow = [f"{v:.6f}" for v in r['best_x']] + ["N/A"]*(max_vars - n)
            w.writerow([r['problem'], r['seed'],
                        _fstr(r['best_f']),
                        r['is_feasible'],
                        f"{r['feas_rate']:.1f}",
                        f"{r['total_cv']:.6e}",
                        f"{r['time_ms']:.0f}"] + xrow)

# 04b. Variable distribution
with open("results/04b_variable_distribution.csv", "w", newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f)
    w.writerow(["Problem", "Variable",
                "Mean_x_all_runs", "Std_x_all_runs",
                "Mean_x_feasible", "Std_x_feasible",
                "Mean_f_feasible", "Std_f_feasible",
                "Var_f_feasible",  "Feas_Rate_pct"])
    for pk in PROBLEM_ORDER:
        p  = PROBLEMS[pk]
        ss = stat_summary(pk)
        for vi, vname in enumerate(p['var_names']):
            mf = ss['var_mean_feas'][vi];  sf = ss['var_std_feas'][vi]
            w.writerow([pk, vname,
                        f"{ss['var_mean_all'][vi]:.6f}",
                        f"{ss['var_std_all'][vi]:.6f}",
                        f"{mf:.6f}" if not np.isnan(mf) else "N/A",
                        f"{sf:.6f}" if not np.isnan(sf) else "N/A",
                        _fstr(ss['mean_f']), _fstr(ss['std_f']),
                        _fstr(ss['var_f']),  f"{ss['feas_rate']:.1f}"])

# 05. Convergence data
with open("results/05_convergence_data.csv", "w", newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f)
    w.writerow(["Step"] + PROBLEM_ORDER)
    for step in range(N_STEPS):
        row = [step + 1]
        for pk in PROBLEM_ORDER:
            fh = main_results[pk]['bo']['fhist']
            row.append(f"{fh[step]:.6f}" if fh is not None else "N/A")
        w.writerow(row)

# 06. Initialization comparison  (updated method label + CDU hyperparams)
with open("results/06_initialization_comparison.csv", "w", newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f)
    w.writerow(["Problem", "Method", "Initialization",
                "Best_K", "Best_rho", "Best_lr",
                "Best_f", "Feasible", "Total_CV",
                "Rel_Gap_wrt_other", "Time_ms"])
    for pk in PROBLEM_ORDER:
        bo = main_results[pk]['bo'];  ip = main_results[pk]['ipopt']
        for tag, init_desc, d, k_, r_, l_ in [
            ("BO+CDU",     "Bayesian TPE + CDU (Optuna)", bo,
             bo.get('best_K','N/A'), bo.get('best_rho','N/A'), bo.get('best_lr','N/A')),
            ("IPOPT_fixed","[1, 1, ..., 1] fixed",        ip, 'N/A','N/A','N/A'),
        ]:
            w.writerow([pk, tag, init_desc, k_, r_, l_,
                        _fstr(d['best_f']), d['feasible'],
                        _fstr(d['total_cv']) if not np.isnan(d['total_cv']) else "N/A",
                        _fstr(d['rel_gap']),
                        f"{d['time_ms']:.0f}"])

print("  OK  All CSVs saved to results/")


# ══════════════════════════════════════════════════════════════════════════════
#  PLOT HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def _sv(v):
    return float(v) if (not np.isinf(v) and not np.isnan(v)) else 0.

def _style_ax(ax):
    ax.tick_params(colors='#333333', labelcolor='#333333')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444444')
    ax.title.set_color('#111111')
    ax.xaxis.label.set_color('#111111')
    ax.yaxis.label.set_color('#111111')

labels  = [DISPLAY_NAMES[pk] for pk in PROBLEM_ORDER]
x_pos   = np.arange(len(PROBLEM_ORDER))
bw      = 0.35
steps   = np.arange(1, N_STEPS + 1)


# ══════════════════════════════════════════════════════════════════════════════
#  FIGURE 01 — CONVERGENCE CURVES
# ══════════════════════════════════════════════════════════════════════════════
print("\n  >  Generating figures ...")

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle(
    "CDU Convergence: BO+CDU Best Trial f(x_K) vs IPOPT Solution",
    fontsize=12, fontweight='bold', y=1.01)
axes_flat = axes.flatten()

for idx, pk in enumerate(PROBLEM_ORDER):
    ax    = axes_flat[idx]
    fhist = main_results[pk]['bo']['fhist']
    ip_f  = main_results[pk]['ipopt']['best_f']
    bK    = main_results[pk]['bo'].get('best_K', '?')
    br    = main_results[pk]['bo'].get('best_rho', '?')

    if fhist is not None:
        ax.plot(steps, fhist, color=C_BO, lw=1.8,
                label=f'BO+CDU (K={bK}, ρ={br:.0e})')
        ax.scatter([steps[-1]], [fhist[-1]], color=C_BO, s=45, zorder=5)

    if not np.isinf(ip_f):
        ax.axhline(ip_f, color=C_FIXED, lw=1.4, ls='--',
                   label=f'IPOPT fixed ({ip_f:.4g})')

    ax.set_title(DISPLAY_NAMES[pk], pad=5)
    ax.set_xlabel("Gradient Step")
    ax.set_ylabel("Objective f(x_K)")
    ax.legend(loc='best', fontsize=7)
    ax.grid(True, **GRID_KW)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(5, integer=True))
    _style_ax(ax)

axes_flat[-1].set_visible(False)
plt.tight_layout(h_pad=2.0, w_pad=1.2)
plt.savefig("figures/01_convergence_curves.png")
plt.close()
print("      -> figures/01_convergence_curves.png")


# ══════════════════════════════════════════════════════════════════════════════
#  FIGURE 02 — RELATIVE OPTIMALITY GAP
# ══════════════════════════════════════════════════════════════════════════════
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Relative Optimality Gap: BO+CDU vs IPOPT",
             fontsize=12, fontweight='bold')

rel_gap_bo_main = [_sv(main_results[pk]['bo']['rel_gap'])    for pk in PROBLEM_ORDER]
rel_gap_fi_main = [_sv(main_results[pk]['ipopt']['rel_gap']) for pk in PROBLEM_ORDER]

stat_gap_means, stat_gap_stds = [], []
for pk in PROBLEM_ORDER:
    ipf  = main_results[pk]['ipopt']['best_f']
    gaps = [relative_gap(r['best_f'], ipf) for r in stat_results[pk]]
    gaps = [g for g in gaps if not np.isnan(g)]
    stat_gap_means.append(np.mean(gaps) if gaps else 0.)
    stat_gap_stds.append(np.std(gaps)   if gaps else 0.)

ax1.bar(x_pos - bw/2, rel_gap_bo_main, bw, color=C_BO,    alpha=0.85,
        label='BO+CDU', edgecolor='#333333', linewidth=0.5)
ax1.bar(x_pos + bw/2, rel_gap_fi_main, bw, color=C_FIXED, alpha=0.85,
        label='IPOPT fixed [1,...,1]', edgecolor='#333333', linewidth=0.5)
ax1.set_yscale('symlog', linthresh=1e-4)
ax1.set_xticks(x_pos);  ax1.set_xticklabels(labels, rotation=30, ha='right')
ax1.set_title("|f_a - f_b| / |f_b|  (each vs the other)", pad=6)
ax1.set_ylabel("Relative Optimality Gap (symlog)")
ax1.legend();  ax1.grid(True, axis='y', **GRID_KW);  _style_ax(ax1)

ax2.bar(x_pos, stat_gap_means, bw*1.2, color=C_BO, alpha=0.85,
        label=f'BO+CDU mean ({N_STAT_RUNS} runs)', edgecolor='#333333', linewidth=0.5)
ax2.errorbar(x_pos, stat_gap_means, yerr=stat_gap_stds,
             fmt='none', color='#333333', capsize=5, lw=1.3, label='std dev')
ax2.set_yscale('symlog', linthresh=1e-4)
ax2.set_xticks(x_pos);  ax2.set_xticklabels(labels, rotation=30, ha='right')
ax2.set_title(f"BO+CDU Rel. Gap: Mean +/- Std ({N_STAT_RUNS} runs vs IPOPT)", pad=6)
ax2.set_ylabel("Relative Gap (mean +/- std, symlog)")
ax2.legend();  ax2.grid(True, axis='y', **GRID_KW);  _style_ax(ax2)

plt.tight_layout()
plt.savefig("figures/02_relative_gap.png")
plt.close()
print("      -> figures/02_relative_gap.png")


# ══════════════════════════════════════════════════════════════════════════════
#  FIGURE 03 — FEASIBILITY RELIABILITY
# ══════════════════════════════════════════════════════════════════════════════
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("CDU Feasibility Reliability Analysis", fontsize=12, fontweight='bold')

feas_rates = [stat_summary(pk)['feas_rate'] for pk in PROBLEM_ORDER]
tcv_bo     = [_sv(main_results[pk]['bo']['total_cv'])    for pk in PROBLEM_ORDER]
tcv_fi     = [_sv(main_results[pk]['ipopt']['total_cv']) for pk in PROBLEM_ORDER]

bar_colors = ['#2166ac' if r >= 80 else '#fc8d59' if r >= 50 else '#d73027'
              for r in feas_rates]
bars = ax1.bar(x_pos, feas_rates, color=bar_colors, alpha=0.85, width=0.55,
               edgecolor='#333333', linewidth=0.6)
ax1.axhline(100, color='#444444', lw=0.8, ls=':', alpha=0.6)
for bar, val in zip(bars, feas_rates):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 1.2,
             f"{val:.0f}%", ha='center', va='bottom',
             fontsize=8, fontweight='bold', color='#111111')
ax1.set_xticks(x_pos);  ax1.set_xticklabels(labels, rotation=30, ha='right')
ax1.set_ylim(0, 115)
ax1.set_title(f"BO+CDU Feasibility Rate ({N_STAT_RUNS} runs)", pad=6)
ax1.set_ylabel("Feasibility Rate (%)")
ax1.grid(True, axis='y', **GRID_KW)
from matplotlib.patches import Patch
ax1.legend(handles=[
    Patch(facecolor='#2166ac', label='Rate >= 80%'),
    Patch(facecolor='#fc8d59', label='50% <= Rate < 80%'),
    Patch(facecolor='#d73027', label='Rate < 50%')], fontsize=8, loc='lower right')
_style_ax(ax1)

ax2.bar(x_pos - bw/2, tcv_bo, bw, color=C_BO,    alpha=0.85,
        label='BO+CDU', edgecolor='#333333', linewidth=0.5)
ax2.bar(x_pos + bw/2, tcv_fi, bw, color=C_FIXED, alpha=0.85,
        label='IPOPT fixed [1,...,1]', edgecolor='#333333', linewidth=0.5)
ax2.set_yscale('symlog', linthresh=1e-8)
ax2.set_xticks(x_pos);  ax2.set_xticklabels(labels, rotation=30, ha='right')
ax2.set_title("Total Constraint Violation  Σ max(gᵢ, 0)", pad=6)
ax2.set_ylabel("Total CV (symlog scale)")
ax2.legend();  ax2.grid(True, axis='y', **GRID_KW);  _style_ax(ax2)

plt.tight_layout()
plt.savefig("figures/03_feasibility_analysis.png")
plt.close()
print("      -> figures/03_feasibility_analysis.png")


# ══════════════════════════════════════════════════════════════════════════════
#  FIGURE 04 — COMPARISON SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
fig, axes4 = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("BO+CDU vs IPOPT Fixed Init — Overall Comparison",
             fontsize=12, fontweight='bold')

bo_f_vals = [_sv(main_results[pk]['bo']['best_f'])    for pk in PROBLEM_ORDER]
fi_f_vals = [_sv(main_results[pk]['ipopt']['best_f']) for pk in PROBLEM_ORDER]
bo_t_vals = [main_results[pk]['bo']['time_ms']        for pk in PROBLEM_ORDER]
fi_t_vals = [main_results[pk]['ipopt']['time_ms']     for pk in PROBLEM_ORDER]

for ax, ybo, yfi, title, ylabel, yscale in [
    (axes4[0], bo_f_vals,       fi_f_vals,       "Best Objective Value",    "f(x)",         None),
    (axes4[1], rel_gap_bo_main, rel_gap_fi_main, "Relative Optimality Gap", "Relative Gap", 'symlog'),
    (axes4[2], bo_t_vals,       fi_t_vals,       "Wall-Clock Time (ms)",    "Time (ms)",    None),
]:
    ax.bar(x_pos - bw/2, ybo, bw, color=C_BO,    alpha=0.85,
           label='BO+CDU', edgecolor='#333333', linewidth=0.5)
    ax.bar(x_pos + bw/2, yfi, bw, color=C_FIXED, alpha=0.85,
           label='IPOPT fixed [1,...,1]', edgecolor='#333333', linewidth=0.5)
    if yscale:
        ax.set_yscale(yscale, linthresh=1e-4)
    ax.set_xticks(x_pos);  ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_title(title, pad=6);  ax.set_ylabel(ylabel)
    ax.legend();  ax.grid(True, axis='y', **GRID_KW);  _style_ax(ax)

plt.tight_layout()
plt.savefig("figures/04_comparison_summary.png")
plt.close()
print("      -> figures/04_comparison_summary.png")


# ══════════════════════════════════════════════════════════════════════════════
#  FIGURE 05 — DECISION VARIABLE DISTRIBUTION
# ══════════════════════════════════════════════════════════════════════════════
fig, axes5 = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle(
    f"Decision Variable Distribution across {N_STAT_RUNS} CDU Runs (BO+CDU)",
    fontsize=12, fontweight='bold', y=1.01)
axes5_flat = axes5.flatten()

for idx, pk in enumerate(PROBLEM_ORDER):
    ax     = axes5_flat[idx]
    p      = PROBLEMS[pk]
    ss     = stat_summary(pk)
    vnames = p['var_names']
    nv     = p['n_vars']
    vx     = np.arange(nv)

    ax.errorbar(vx - 0.12, ss['var_mean_all'], yerr=ss['var_std_all'],
                fmt='o', color=C_BO, capsize=5, lw=1.4, ms=5,
                label='All runs (mean +/- std)')
    if not np.all(np.isnan(ss['var_mean_feas'])):
        ax.errorbar(vx + 0.12, ss['var_mean_feas'], yerr=ss['var_std_feas'],
                    fmt='s', color=C_FIXED, capsize=5, lw=1.4, ms=5,
                    label='Feasible only (mean +/- std)')

    ax.set_xticks(vx);  ax.set_xticklabels(vnames)
    ax.set_title(DISPLAY_NAMES[pk], pad=5)
    ax.set_ylabel("Variable Value");  ax.set_xlabel("Decision Variable")
    ax.legend(fontsize=8);  ax.grid(True, axis='y', **GRID_KW);  _style_ax(ax)

    mf = ss['mean_f'];  sf = ss['std_f']
    label_txt = (f"f: {mf:.4g} +/- {sf:.3g}" if not np.isnan(mf)
                 else "No feasible solutions")
    ax.text(0.97, 0.97, label_txt, transform=ax.transAxes,
            fontsize=7.5, ha='right', va='top', color='#111111',
            bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                      edgecolor='#999999', alpha=0.9))

axes5_flat[-1].set_visible(False)
plt.tight_layout(h_pad=2.5, w_pad=1.5)
plt.savefig("figures/05_variable_distribution.png")
plt.close()
print("      -> figures/05_variable_distribution.png")


# ══════════════════════════════════════════════════════════════════════════════
#  TERMINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*85)
print("  MAIN RESULTS — BO+CDU vs IPOPT Fixed Init")
print("  CDU Loss: log((f+M)²+ε) + λ_K·relu(g) + (ρ/2)‖relu(g)‖²")
print("  Feasibility: g_i(x) ≤ 0 strict for all i")
print("="*85)
print(f"  {'Problem':<22} {'BO+CDU f':>12} {'IPOPT f':>12}  "
      f"{'Rel gap CDU':>12}  {'Rel gap IP':>12}  {'CDU':>5}  {'IP':>5}  "
      f"{'K':>4}  {'rho':>8}")
print("-"*85)
for pk in PROBLEM_ORDER:
    bo = main_results[pk]['bo'];  ip = main_results[pk]['ipopt']
    print(f"  {DISPLAY_NAMES[pk]:<22}"
          f" {_sv(bo['best_f']):>12.4f}"
          f" {_sv(ip['best_f']):>12.4f}"
          f"  {_sv(bo['rel_gap']):>12.4f}"
          f"  {_sv(ip['rel_gap']):>12.4f}"
          f"  {'Y' if bo['feasible'] else 'N':>5}"
          f"  {'Y' if ip['feasible'] else 'N':>5}"
          f"  {str(bo.get('best_K','?')):>4}"
          f"  {str(bo.get('best_rho','?')):>8}")
print("="*85)

print(f"\n  STATISTICAL ROBUSTNESS  (BO+CDU, N={N_STAT_RUNS} runs)")
print(f"  {'Problem':<22} {'mean f':>12} {'std f':>10} {'var f':>12}  {'Feas%':>7}")
print("-"*68)
for pk in PROBLEM_ORDER:
    ss = stat_summary(pk)
    print(f"  {DISPLAY_NAMES[pk]:<22}"
          f" {_sv(ss['mean_f']):>12.5g}"
          f" {_sv(ss['std_f']):>10.4g}"
          f" {_sv(ss['var_f']):>12.4g}"
          f"  {ss['feas_rate']:>6.1f}%")
print("="*85)

print("\n  DECISION VARIABLE DISTRIBUTION  (BO+CDU, feasible runs)")
print(f"  {'Problem':<22} {'Var':>5}  {'mean (all)':>12}  {'std (all)':>10}  "
      f"{'mean (feas)':>12}  {'std (feas)':>10}")
print("-"*80)
for pk in PROBLEM_ORDER:
    ss = stat_summary(pk);  p = PROBLEMS[pk]
    for vi, vname in enumerate(p['var_names']):
        mf = ss['var_mean_feas'][vi];  sf = ss['var_std_feas'][vi]
        print(f"  {DISPLAY_NAMES[pk]:<22} {vname:>5}  "
              f"{ss['var_mean_all'][vi]:>12.5g}  "
              f"{ss['var_std_all'][vi]:>10.4g}  "
              f"{(mf if not np.isnan(mf) else float('nan')):>12.5g}  "
              f"{(sf if not np.isnan(sf) else float('nan')):>10.4g}")
print("="*85)
print("\n  All figures  -> figures/")
print("  All CSVs     -> results/")
print("\n  Done")

  BO+NN+CDU vs IPOPT (fixed init) — Journal Benchmark Suite
  CDU Loss: log((f+M)²+ε) + λ_K·relu(g) + (ρ/2)‖relu(g)‖²
  Optuna tunes: K (unrolling depth), ρ (aug. Lagrangian), lr, z
  N_TRIALS_MAIN=1000  N_STEPS=150
  CDU_K_CHOICES=[3, 5, 8]  RHO_CHOICES=[100.0, 1000.0, 10000.0]
  LR_CHOICES=[0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05]
  Feasibility: strict g_i(x) <= 0.0 for all i

  Pre-compiling 378 JIT graphs (problem × K × ρ × lr) ...
    OK  Branin
    OK  Himmelblau
    OK  Constrained Rosenbrock
    OK  Pressure Vessel
    OK  G04 Gear Train
    OK  G06 Global Opt.
    OK  Welded Beam


---------------------------------------------------------------------
  [Branin]  Branin
---------------------------------------------------------------------
  >  Main BO+CDU (1000 trials) ...


  Branin                          : 100%|██████████████████████████████| 1000/1000 [00:31<00:00, 31.28trial/s, K=5, best_f=0.3979, feasible=758/1000, rho=1e+02]


  OK  f=0.39789  feasible=True  feas_rate=76%  time=33268ms
       Best CDU trial: K=3  ρ=1e+02  lr=0.0001
  >  IPOPT fixed [1,...,1] ...
  XX  IPOPT: CasADi not available or solver failed.
  >  Statistical analysis (30 × 1000 trials) ...


  Stats: Branin                   : 100%|████████████████████████████| 30/30 [13:39<00:00, 27.32s/run]


  OK  feasible=30/30  mean_f=0.39789  std_f=0.00000

---------------------------------------------------------------------
  [Himmelblau]  Himmelblau
---------------------------------------------------------------------
  >  Main BO+CDU (1000 trials) ...


...

  Stats: G04 Gear Train           : 100%|████████████████████████████| 30/30 [24:11<00:00, 48.37s/run]


  OK  feasible=30/30  mean_f=-30572.30505  std_f=54.28616

---------------------------------------------------------------------
  [G06]  G06 Global Opt.
---------------------------------------------------------------------
  >  Main BO+CDU (1000 trials) ...


  G06 Global Opt.                 : 100%|██████████████████████████████| 1000/1000 [00:29<00:00, 33.56trial/s, K=3, best_f=-6944.0935, feasible=181/1000, rho=1e+02]


  OK  f=-6944.09350  feasible=True  feas_rate=18%  time=30043ms
       Best CDU trial: K=8  ρ=1e+02  lr=0.01
  >  IPOPT fixed [1,...,1] ...
  XX  IPOPT: CasADi not available or solver failed.
  >  Statistical analysis (30 × 1000 trials) ...


  Stats: G06 Global Opt.          :   0%|                            | 0/30 [00:00<?, ?run/s]